# Superstore — Shipping & Regional Analysis

A reproducible Python companion to the Power BI project. The analysis focuses on regional profitability, shipping-mode performance, delivery time, and late Standard Class shipments.

**Dataset:** Sample Superstore (9,994 order lines)  
**Period:** 2014–2017  
**Regions:** Central, East, South, West  
**Shipping modes:** Same Day, First Class, Second Class, Standard Class


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "../data/Sample - Superstore.csv"
df = pd.read_csv(DATA_PATH)

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])
df["Days to Ship"] = (df["Ship Date"] - df["Order Date"]).dt.days

print(f"Rows: {len(df):,}")
print(f"Date range: {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
df.head()


## 1. Regional performance

The core business question is whether regional differences are driven by scale, shipping mix, or profitability.


In [ ]:
regional = (
    df.groupby("Region")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
      .assign(Margin=lambda x: x["Profit"] / x["Sales"] * 100)
      .sort_values("Sales", ascending=False)
)
regional.round(2)


In [ ]:
ax = regional["Sales"].sort_values().plot(kind="bar", figsize=(8,4), title="Sales by Region")
ax.set_xlabel("Sales ($)")
ax.set_ylabel("")
plt.tigght_layout()
plt.show()


### Regional takeaway

The supplied project report identifies **Central** as the key profitability issue: it generates about **$501K in sales but only 7.92% margin**, versus **14.94% in West**. The gap is material because Central is a large market rather than a small tail region.


## 2. Shipping mode × region


In [ ]:
ship_mix = (
    df.groupby(["Region", "Ship Mode"])["Sales"]
      .sum()
      .unstack(fill_value=0)
)
ship_mix.round(0)


In [ ]:
ship_days = (
    df.groupby(["Ship Mode", "Region"])["Days to Ship"]
      .mean()
      .unstack()
)
ship_days.round(2)


In [ ]:
ax = ship_days.plot(kind="bar", figsize=(10,5), title="Average Days to Ship by Mode and Region")
ax.set_ylabel("Average calendar days")
ax.set_xlabel("Ship Mode")
plt.xticks(rotation=0)
plt.tight_layout()
plt.showo)


The project report notes that average shipping times are remarkably similar across regions. Standard Class is around five days everywhere, which weakens the case that logistics are the main explanation for Central's margin gap.


## 3. Late Standard Class shipments

The project defines a late Standard Class shipment as taking **more than five calendar days**.


In [ ]:
late_standard = (
    df.loc[(df["Ship Mode"] == "Standard Class") & (df["Days to Ship"] > 5)]
      .groupby("Region")
      .size()
      .sort_values(ascending=False)
)
late_standard


In [ ]:
ax = late_standard.sort_values().plot(kind="bar", figsize=(8,4), title="Late Standard Class Shipments (>5 days)")
ax.set_ylabel("Order lines")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 4. Profit margin by shipping mode and region


In [ ]:
margin_matrix = (
    df.groupby(["Region", "Ship Mode"])/[["Sales", "Profit"]]
      .sum()
      .assign(Margin=lambda x: x["Profit"] / x["Sales"] * 100)
      ["Margin"]
      .unstack()
)
margin_matrix.round(2)


## 5. Interpretation

The analysis supports the same business framing as the Power BI report:

- **Central is the main regional profitability concern.**
- **Standard Class dominates sales across regions.**
- **Average shipping times are close across regions**, so the Central margin gap should not automatically be attributed to logistics.
- **Late shipments exist at meaningful volume**, with West highest in count largely because it is the largest region by order volume.
- The stronger hypothesis is **commercial margin leakage — especially discounting/pricing — rather than a fundamentally different shipping operation**.

This notebook is deliberately descriptive. It does not claim causal identification; the recommendations in the Power BI report should be treated as business hypotheses to validate with controlled pricing/discount experiments.


## 6. Portfolio note

This notebook is a cleaned reproducibility layer for the broader Power BI group project. The primary interactive deliverable is the Power BI dashboard, while the supplied HTML dashboard provides a lightweight web presentation of the shipping/regional findings.
